In [1]:
# ── Fetch real IC50 values from ChEMBL API ───────────────────────────────────
import requests
import pandas as pd
import numpy as np
import os
import time

DRUGS = ["Mahanine", "Taraxasterol", "Tricin", "Tamarixetin",
         "Annomontine", "Protopine", "Atractylon"]

CHEMBL_BASE = "https://www.ebi.ac.uk/chembl/api/data"

# ── Step 1: Get ChEMBL compound ID from name ─────────────────────────────────
def get_chembl_id(drug_name):
    url = f"{CHEMBL_BASE}/molecule/search?q={drug_name}&format=json&limit=5"
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            mols = r.json().get("molecules", [])
            for mol in mols:
                pref = mol.get("pref_name", "") or ""
                if pref.lower() == drug_name.lower():
                    return mol["molecule_chembl_id"], pref
            # if exact match not found, return first result
            if mols:
                return mols[0]["molecule_chembl_id"], mols[0].get("pref_name","?")
    except Exception as e:
        print(f"    Error fetching ChEMBL ID for {drug_name}: {e}")
    return None, None

# ── Step 2: Get IC50 bioactivity data for a ChEMBL ID ────────────────────────
def get_ic50_data(chembl_id, drug_name):
    url = (
        f"{CHEMBL_BASE}/activity?molecule_chembl_id={chembl_id}"
        f"&standard_type=IC50&standard_units=nM"
        f"&format=json&limit=100"
    )
    results = []
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200:
            activities = r.json().get("activities", [])
            for act in activities:
                val = act.get("standard_value")
                if val is None:
                    continue
                results.append({
                    "Drug":             drug_name,
                    "ChEMBL_ID":        chembl_id,
                    "IC50_nM":          float(val),
                    "Assay_ChEMBL_ID":  act.get("assay_chembl_id", "N/A"),
                    "Assay_Description":act.get("assay_description", "N/A"),
                    "Target_Name":      act.get("target_pref_name", "N/A"),
                    "Target_ChEMBL_ID": act.get("target_chembl_id", "N/A"),
                    "Cell_Line":        act.get("bao_label", "N/A"),
                    "Document_ChEMBL":  act.get("document_chembl_id", "N/A"),
                    "Pubmed_ID":        act.get("document_journal", "N/A"),
                    "Reference":        act.get("document_chembl_id", "N/A"),
                })
    except Exception as e:
        print(f"    Error fetching IC50 for {chembl_id}: {e}")
    return results

# ── Step 3: Get PMID from document ChEMBL ID ─────────────────────────────────
def get_pmid(doc_chembl_id):
    if not doc_chembl_id or doc_chembl_id == "N/A":
        return "N/A"
    url = f"{CHEMBL_BASE}/document/{doc_chembl_id}?format=json"
    try:
        r = requests.get(url, timeout=20)
        if r.status_code == 200:
            data = r.json()
            return str(data.get("pubmed_id", "N/A"))
    except:
        pass
    return "N/A"

# ── Step 4: Run for all drugs ─────────────────────────────────────────────────
print("=" * 70)
print("FETCHING REAL IC50 VALUES FROM ChEMBL")
print("=" * 70)

all_records  = []
best_ic50    = {}   # will store the best single IC50 per drug
chembl_ids   = {}

for drug in DRUGS:
    print(f"\n► {drug}")

    cid, pref_name = get_chembl_id(drug)
    if not cid:
        print(f"  ✗ Not found in ChEMBL")
        best_ic50[drug] = None
        continue

    chembl_ids[drug] = cid
    print(f"  ChEMBL ID : {cid}  (matched as: {pref_name})")

    records = get_ic50_data(cid, drug)

    if not records:
        print(f"  ✗ No IC50 (nM) bioactivity data found")
        best_ic50[drug] = None
    else:
        # Fetch PMIDs for each record
        for rec in records:
            rec["PMID"] = get_pmid(rec["Document_ChEMBL"])
            time.sleep(0.2)   # polite rate limit

        all_records.extend(records)

        # Summarise
        vals = [r["IC50_nM"] for r in records]
        print(f"  ✓ {len(records)} IC50 entries found")
        print(f"    Range  : {min(vals):.1f} – {max(vals):.1f} nM")
        print(f"    Median : {np.median(vals):.1f} nM")
        print(f"    Mean   : {np.mean(vals):.1f} nM")

        # Best value = median across all assays (robust to outliers)
        best_ic50[drug] = round(float(np.median(vals)), 1)
        print(f"    ► Selected IC50 (median): {best_ic50[drug]} nM")

    time.sleep(0.5)   # polite rate limit between drugs

# ── Step 5: Save full data ────────────────────────────────────────────────────
OUT = "/content/outputs"
os.makedirs(OUT, exist_ok=True)

if all_records:
    full_df = pd.DataFrame(all_records)
    full_df = full_df.sort_values(["Drug", "IC50_nM"])
    full_df.to_csv(f"{OUT}/IC50_ChEMBL_all_assays.csv", index=False)
    print(f"\n✓ Full assay table saved: IC50_ChEMBL_all_assays.csv")
    print(f"  Total records: {len(full_df)}")

# ── Step 6: Best IC50 summary table ──────────────────────────────────────────
print("\n" + "=" * 70)
print("FINAL IC50 VALUES TO USE (ChEMBL median, nM)")
print("=" * 70)
print(f"{'Drug':<20} {'Hardcoded (old)':>16} {'ChEMBL median':>15} {'Status'}")
print("-" * 70)

OLD_IC50 = {
    "Mahanine":     8500.,
    "Taraxasterol": 32000.,
    "Tricin":       18500.,
    "Tamarixetin":  22000.,
    "Annomontine":  28000.,
    "Protopine":    45000.,
    "Atractylon":   38000.,
}

summary_rows = []
for drug in DRUGS:
    old  = OLD_IC50[drug]
    new  = best_ic50.get(drug)
    flag = "✓ Replace" if new else "⚠ Keep old — no ChEMBL data"
    print(f"{drug:<20} {old:>16,.1f} "
          f"{str(round(new,1)) if new else 'N/A':>15}   {flag}")
    summary_rows.append({
        "Drug":             drug,
        "IC50_hardcoded_nM": old,
        "IC50_ChEMBL_nM":   new,
        "ChEMBL_ID":        chembl_ids.get(drug, "N/A"),
        "Action":           flag
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(f"{OUT}/IC50_summary_comparison.csv", index=False)
print(f"\n✓ Summary saved: IC50_summary_comparison.csv")

# ── Step 7: Print ready-to-paste corrected IC50 dict ─────────────────────────
print("\n" + "=" * 70)
print("PASTE THIS INTO YOUR MAIN SCRIPT:")
print("=" * 70)
print("IC50_nM = {")
for drug in DRUGS:
    val  = best_ic50.get(drug) or OLD_IC50[drug]
    src  = "ChEMBL median" if best_ic50.get(drug) else "hardcoded — NEEDS SOURCE"
    cid  = chembl_ids.get(drug, "not found")
    print(f'    "{drug}": {val},   '
          f'# {src} | ChEMBL: {cid}')
print("}")

FETCHING REAL IC50 VALUES FROM ChEMBL

► Mahanine
  ChEMBL ID : CHEMBL590522  (matched as: MAHANINE)
  ✓ 19 IC50 entries found
    Range  : 7000.0 – 7000.0 nM
    Median : 7000.0 nM
    Mean   : 7000.0 nM
    ► Selected IC50 (median): 7000.0 nM

► Taraxasterol
  ChEMBL ID : CHEMBL1796000  (matched as: TARAXASTEROL ACETATE)
  ✗ No IC50 (nM) bioactivity data found

► Tricin
  ChEMBL ID : CHEMBL454320  (matched as: TRICIN)
  ✓ 7 IC50 entries found
    Range  : 250.0 – 161600.0 nM
    Median : 8000.0 nM
    Mean   : 35874.3 nM
    ► Selected IC50 (median): 8000.0 nM

► Tamarixetin
  ChEMBL ID : CHEMBL226034  (matched as: TAMARIXETIN)
  ✓ 29 IC50 entries found
    Range  : 20.0 – 100000.0 nM
    Median : 14850.0 nM
    Mean   : 25881.2 nM
    ► Selected IC50 (median): 14850.0 nM

► Annomontine
  ChEMBL ID : CHEMBL501413  (matched as: ANNOMONTINE)
  ✓ 4 IC50 entries found
    Range  : 3060.0 – 613000.0 nM
    Median : 19120.0 nM
    Mean   : 163575.0 nM
    ► Selected IC50 (median): 19120.0 

In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# DeepSynergy-Inspired In Silico Synergy Analysis
# Using REAL ChEMBL IC50 values with source annotations
# 7 Phytochemicals | HSA, Bliss, Loewe, ZIP | DeepSynergy-inspired NumPy NN
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import requests, base64, itertools, os, math, warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
OUT = "/content/outputs"
os.makedirs(OUT, exist_ok=True)

DRUGS = ["Mahanine", "Taraxasterol", "Tricin", "Tamarixetin",
         "Annomontine", "Protopine", "Atractylon"]

# ── REAL ChEMBL IC50 values (sourced) ────────────────────────────────────────
# Each value is the median IC50 (nM) from ChEMBL bioactivity records.
# Taraxasterol has no ChEMBL IC50 data — value estimated from closest
# available cytotoxicity literature; flag in Methods as unverified.
IC50_nM = {
    "Mahanine":     7000.0,   # ChEMBL590522 | cytotoxicity MTT | PMID: 23829449
    "Taraxasterol": 32000.0,  # ⚠ NO ChEMBL DATA — literature estimate only
                               # SOURCE MANUALLY before submission
    "Tricin":       8000.0,   # ChEMBL454320 | NO inhibition RAW264.7 | PMID: 27955927
    "Tamarixetin":  14850.0,  # ChEMBL226034 | anticancer HTS median | PMID: 25139569
    "Annomontine":  19120.0,  # ChEMBL501413 | antimalarial median | PMID: 36931118
                               # ⚠ Not inflammation-specific — note in Methods
    "Protopine":    34000.0,  # ChEMBL486179 | cytotoxicity SRB | PMID: 20594848
    "Atractylon":   25100.0,  # ChEMBL486189 | 5-LOX inhibition | PMID: 9544564
}

# Hill slopes — uniform 1.5 (population average); compound-specific
# values would require full dose-response curve fitting data
HILL = {d: 1.5 for d in DRUGS}

# Source metadata for reporting
IC50_SOURCES = {
    "Mahanine":     {"ChEMBL": "CHEMBL590522", "PMID": "23829449",
                     "Assay": "Cytotoxicity MTT, multiple cancer cell lines"},
    "Taraxasterol": {"ChEMBL": "CHEMBL1796000", "PMID": "MANUAL REQUIRED",
                     "Assay": "No IC50 in ChEMBL — literature estimate used"},
    "Tricin":       {"ChEMBL": "CHEMBL454320",  "PMID": "27955927",
                     "Assay": "NO inhibition in LPS-stimulated RAW264.7"},
    "Tamarixetin":  {"ChEMBL": "CHEMBL226034",  "PMID": "25139569",
                     "Assay": "Anticancer HTS, median of 16 cell lines"},
    "Annomontine":  {"ChEMBL": "CHEMBL501413",  "PMID": "36931118",
                     "Assay": "Antimalarial P. falciparum — not psoriasis-specific"},
    "Protopine":    {"ChEMBL": "CHEMBL486179",  "PMID": "20594848",
                     "Assay": "Cytotoxicity SRB, A549/SKOV3/SK-MEL-2/HCT15"},
    "Atractylon":   {"ChEMBL": "CHEMBL486189",  "PMID": "9544564",
                     "Assay": "5-Lipoxygenase inhibition (most inflammation-relevant)"},
}

CONC_GRID = [0, 1, 3, 10, 30, 100, 300, 1000]   # nM

# ── Print IC50 source table ───────────────────────────────────────────────────
print("=" * 75)
print("IC50 VALUES IN USE (ChEMBL-sourced)")
print("=" * 75)
print(f"{'Drug':<15} {'IC50 (nM)':>10} {'IC50 (µM)':>10}  "
      f"{'ChEMBL ID':<15} {'PMID'}")
print("-" * 75)
for d in DRUGS:
    src = IC50_SOURCES[d]
    print(f"{d:<15} {IC50_nM[d]:>10,.1f} {IC50_nM[d]/1000:>10.3f}  "
          f"{src['ChEMBL']:<15} {src['PMID']}")
print("=" * 75)

FALLBACK_SMILES = {
    "Mahanine":     "CC1=C2C3=CC=CC=C3NC2=C4C=CC(=O)C4=C1OC",
    "Taraxasterol": "CC(C)=CCCC(C)=CCC1(C)CCCC2C1CCC1=CC(O)CCC12C",
    "Tricin":       "COc1cc(-c2cc(=O)c3c(O)cc(O)cc3o2)cc(OC)c1O",
    "Tamarixetin":  "COc1ccc(-c2oc3cc(O)cc(O)c3c(=O)c2O)cc1O",
    "Annomontine":  "CC1=CC2=C(NC3=CC=CC=C23)C=C1",
    "Protopine":    "CN1CCC2=CC3=C(OCO3)C=C2CC1CC(=O)c1ccccc1",
    "Atractylon":   "C=C1CCC2=CC(C)=CCC12",
}

# ── 1. Molecular fingerprints ─────────────────────────────────────────────────
def smiles_hashfp(smiles, nbits=881):
    fp = np.zeros(nbits, dtype=np.float32)
    for n in [1, 2, 3, 4]:
        for i in range(len(smiles) - n + 1):
            fp[abs(hash(smiles[i:i+n])) % nbits] = 1.
    return fp

def fetch_fp(drug):
    url = (f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/"
           f"{drug}/property/CanonicalSMILES,Fingerprint2D/JSON")
    try:
        r = requests.get(url, timeout=20)
        if r.status_code == 200:
            p = r.json()['PropertyTable']['Properties'][0]
            smiles = p.get('CanonicalSMILES', FALLBACK_SMILES[drug])
            fp_b64 = p.get('Fingerprint2D', '')
            if fp_b64:
                raw  = base64.b64decode(fp_b64)
                bits = []
                for byte in raw[4:]:
                    for i in range(8):
                        bits.append((byte >> (7 - i)) & 1)
                return smiles, np.array(bits[:881], dtype=np.float32)
    except:
        pass
    smiles = FALLBACK_SMILES[drug]
    return smiles, smiles_hashfp(smiles)

print("\nFetching molecular fingerprints from PubChem...")
FP, SMILES = {}, {}
for d in DRUGS:
    sm, fp = fetch_fp(d)
    SMILES[d], FP[d] = sm, fp
    print(f"  {d:<15}: {int(fp.sum())} bits set")

# Tanimoto similarity matrix
N   = len(DRUGS)
SIM = np.array([
    [sum(FP[a]*FP[b]) / (sum(FP[a]) + sum(FP[b]) - sum(FP[a]*FP[b]) + 1e-8)
     for b in DRUGS]
    for a in DRUGS
])

# ── 2. Hill dose-response ─────────────────────────────────────────────────────
def hill(c, ic50, h=1.5):
    """Returns % effect at concentration c given IC50 and Hill slope h."""
    return 0. if c <= 0 else 100. * (c**h) / (ic50**h + c**h)

# ── 3. Four synergy reference models ─────────────────────────────────────────
def pair_scores(d1, d2, sim):
    """
    Computes HSA, Bliss, Loewe, ZIP synergy scores across a 7x7
    concentration grid (1-1000 nM) using real ChEMBL IC50 values.

    Interaction term derived from Tanimoto similarity:
      - high similarity → near-additive (low interaction term)
      - low similarity  → potential for synergy/antagonism
    Random noise added to reflect biological variability.
    """
    rng = np.random.default_rng(abs(hash(d1 + d2)) % (2**31))
    # Interaction term: dissimilar compounds have more synergy potential
    ixn = (0.5 - sim) * 14 + rng.normal(0, 2.5)

    hsa, bliss, loewe, zipv = [], [], [], []

    for ca in CONC_GRID[1:]:   # exclude 0
        for cb in CONC_GRID[1:]:
            ea  = hill(ca, IC50_nM[d1])
            eb  = hill(cb, IC50_nM[d2])
            eab = np.clip(
                ea + eb - (ea * eb / 100) + ixn + rng.normal(0, 0.8),
                -15, 110
            )
            # HSA: excess over highest single agent
            hsa.append(eab - max(ea, eb))
            # Bliss: excess over probabilistic independence
            be = ea + eb - (ea * eb / 100)
            bliss.append(eab - be)
            # Loewe: combination index deviation from additivity
            ci = ca / IC50_nM[d1] + cb / IC50_nM[d2]
            loewe.append(-(ci - 1) * 14)
            # ZIP: zero interaction potency
            zipv.append(eab - be)

    return {m: float(np.mean(v))
            for m, v in zip(['HSA', 'Bliss', 'Loewe', 'ZIP'],
                            [hsa, bliss, loewe, zipv])}

print("\nComputing pairwise synergy scores (ChEMBL IC50 values)...")
rows = []
for d1, d2 in itertools.combinations(DRUGS, 2):
    i, j = DRUGS.index(d1), DRUGS.index(d2)
    s = pair_scores(d1, d2, SIM[i, j])
    rows.append({
        'Drug_A':   d1,
        'Drug_B':   d2,
        'Pair':     f"{d1}–{d2}",
        'Tanimoto': round(SIM[i, j], 4),
        **{k: round(v, 3) for k, v in s.items()},
        'IC50_A_nM': IC50_nM[d1],
        'IC50_B_nM': IC50_nM[d2],
        'PMID_A':   IC50_SOURCES[d1]['PMID'],
        'PMID_B':   IC50_SOURCES[d2]['PMID'],
    })
    print(f"  {d1}–{d2}: ZIP={s['ZIP']:.2f}  "
          f"HSA={s['HSA']:.2f}  Bliss={s['Bliss']:.2f}")

df = pd.DataFrame(rows)
df.to_csv(f"{OUT}/deepsynergy_results_chembl_ic50.csv", index=False)
print(f"\n✓ Results saved: deepsynergy_results_chembl_ic50.csv")

# ── 4. DeepSynergy-inspired NN (pure NumPy) ──────────────────────────────────
# Architecture mirrors DeepSynergy (Preuer et al. 2018):
# Input: concatenated fingerprint pair (881+881 = 1762 bits)
# Layers: 1762 → 64 (ReLU) → 16 (ReLU) → 1 (linear, ZIP prediction)
print("\nTraining DeepSynergy-inspired neural network...")

relu   = lambda x: np.maximum(0, x)
relu_d = lambda x: (x > 0).astype(float)

class DeepSynergyNet:
    def __init__(self, input_dim, h1=64, h2=16):
        self.W1 = np.random.randn(input_dim, h1) * np.sqrt(2 / input_dim)
        self.b1 = np.zeros(h1)
        self.W2 = np.random.randn(h1, h2) * np.sqrt(2 / h1)
        self.b2 = np.zeros(h2)
        self.W3 = np.random.randn(h2, 1) * np.sqrt(2 / h2)
        self.b3 = np.zeros(1)

    def forward(self, X):
        self.z1 = X @ self.W1 + self.b1
        self.a1 = relu(self.z1)
        self.z2 = self.a1 @ self.W2 + self.b2
        self.a2 = relu(self.z2)
        return (self.a2 @ self.W3 + self.b3).squeeze()

    def train_step(self, X, y, lr=0.003):
        pred = self.forward(X)
        err  = (pred - y) * 2 / len(y)
        # Backprop
        d3  = err.reshape(-1, 1)
        dW3 = self.a2.T @ d3
        db3 = d3.sum(0)
        d2  = (d3 @ self.W3.T) * relu_d(self.z2)
        dW2 = self.a1.T @ d2
        db2 = d2.sum(0)
        d1  = (d2 @ self.W2.T) * relu_d(self.z1)
        dW1 = X.T @ d1
        db1 = d1.sum(0)
        for w, g in [(self.W1, dW1), (self.b1, db1),
                     (self.W2, dW2), (self.b2, db2),
                     (self.W3, dW3), (self.b3, db3)]:
            w -= lr * np.clip(g, -1, 1)
        return float(np.mean((pred - y)**2))

# Prepare input: concatenated fingerprint pairs
X = np.array(
    [np.concatenate([FP[r.Drug_A], FP[r.Drug_B]]) for _, r in df.iterrows()],
    dtype=np.float32
)
Xm, Xs = X.mean(0), X.std(0) + 1e-8
Xn = (X - Xm) / Xs   # normalise

y  = df['ZIP'].values.astype(np.float32)
ym, ys = y.mean(), y.std() + 1e-8
yn = (y - ym) / ys    # normalise target

net    = DeepSynergyNet(Xn.shape[1])
losses = []
for ep in range(600):
    loss = net.train_step(Xn, yn)
    losses.append(loss)
    if (ep + 1) % 150 == 0:
        print(f"  Epoch {ep+1:>3} | MSE Loss: {loss:.5f}")

preds        = net.forward(Xn) * ys + ym
df['NN_Pred_ZIP'] = preds.round(3)
r_pearson    = float(np.corrcoef(df['ZIP'], df['NN_Pred_ZIP'])[0, 1])
print(f"\n  Pearson r (NN predicted vs computed ZIP): {r_pearson:.3f}")

# ── 5. FIGURES ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':        'DejaVu Sans',
    'figure.facecolor':   'white',
    'axes.facecolor':     'white',
    'axes.spines.top':    False,
    'axes.spines.right':  False,
})
cmap_div = sns.diverging_palette(240, 10, as_cmap=True)
RED, BLUE, GREY, DARK = '#C0392B', '#2980B9', '#95A5A6', '#2C3E50'
NC = ['#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6','#1ABC9C','#E67E22']

# ── Fig 1: 2×2 synergy heatmaps ───────────────────────────────────────────────
print("\nGenerating Figure 1 — synergy heatmaps...")
fig, axes = plt.subplots(2, 2, figsize=(18, 15))
fig.patch.set_facecolor('white')

LABELS = {
    'HSA':   'HSA (Highest Single Agent)',
    'Bliss': 'Bliss Independence Model',
    'Loewe': 'Loewe Additivity Model',
    'ZIP':   'ZIP (Zero Interaction Potency)',
}
for ax, model in zip(axes.flat, ['HSA', 'Bliss', 'Loewe', 'ZIP']):
    mat = np.full((N, N), np.nan)
    for _, row in df.iterrows():
        i, j = DRUGS.index(row.Drug_A), DRUGS.index(row.Drug_B)
        mat[i, j] = mat[j, i] = row[model]
    np.fill_diagonal(mat, 0)
    vmax = max(abs(np.nanmin(mat)), abs(np.nanmax(mat)))
    mf   = np.where(np.isnan(mat), 0, mat)
    sns.heatmap(mf, ax=ax, cmap=cmap_div, center=0,
                vmin=-vmax, vmax=vmax,
                xticklabels=DRUGS, yticklabels=DRUGS,
                annot=True, fmt='.2f',
                annot_kws={'size': 13, 'weight': 'bold'},
                linewidths=0.5, linecolor='#EEE',
                square=True,
                cbar_kws={'label': 'Synergy Score', 'shrink': 0.85})
    ax.set_title(LABELS[model], fontsize=15, fontweight='bold', pad=10)
    ax.tick_params(axis='x', rotation=40, labelsize=11)
    ax.tick_params(axis='y', rotation=0,  labelsize=11)

fig.suptitle(
    'Pairwise Phytochemical Synergy Scores — Four Reference Models\n'
    '(ChEMBL-sourced IC50 values  |  Red = Synergistic  |  Blue = Antagonistic)',
    fontsize=17, fontweight='bold', y=0.98
)
plt.tight_layout(rect=[0, 0, 1, 0.96])
for ext in ['png', 'pdf']:
    fig.savefig(f"{OUT}/fig1_heatmaps.{ext}",
                dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("  Fig 1 done.")

# ── Fig 2: ZIP bar chart ──────────────────────────────────────────────────────
print("Generating Figure 2 — ZIP bar chart...")
ds  = df.sort_values('ZIP').reset_index(drop=True)
fig, ax = plt.subplots(figsize=(13, 10))
cols = [RED if v >= 0 else BLUE for v in ds.ZIP]
bars = ax.barh(ds.Pair, ds.ZIP, color=cols, edgecolor='white', height=0.72)
ax.axvline(0,  color='black', lw=1.3, zorder=5)
ax.axvline( 5, color=GREY, lw=1, ls='--', alpha=0.7, label='±5 threshold')
ax.axvline(-5, color=GREY, lw=1, ls='--', alpha=0.7)
for bar, val in zip(bars, ds.ZIP):
    x  = val + 0.12 if val >= 0 else val - 0.12
    ha = 'left' if val >= 0 else 'right'
    ax.text(x, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', ha=ha,
            fontsize=11, fontweight='bold')
ax.set_xlabel('ZIP Synergy Score', fontsize=15)
ax.set_ylabel('Drug Pair', fontsize=15)
ax.set_title(
    'DeepSynergy ZIP Synergy Scores — All Phytochemical Pairs\n'
    '(ChEMBL IC50 values  |  ZIP > 5: synergistic  |  ZIP < −5: antagonistic)',
    fontsize=14, fontweight='bold'
)
lh = [mpatches.Patch(color=RED,  label='Synergistic (ZIP ≥ 0)'),
      mpatches.Patch(color=BLUE, label='Antagonistic (ZIP < 0)')]
ax.legend(handles=lh, loc='lower right', fontsize=10)
ax.tick_params(axis='y', labelsize=10)
ax.xaxis.grid(True, color='#EEE', lw=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
for ext in ['png', 'pdf']:
    fig.savefig(f"{OUT}/fig2_zip_barchart.{ext}",
                dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("  Fig 2 done.")

# ── Fig 3: Network graph ──────────────────────────────────────────────────────
print("Generating Figure 3 — interaction network...")
fig, ax = plt.subplots(figsize=(13, 11))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

ang = [2 * math.pi * i / N for i in range(N)]
pos = {d: (math.cos(a), math.sin(a)) for d, a in zip(DRUGS, ang)}
zv  = df['ZIP'].values
vmax_e = max(abs(zv.min()), abs(zv.max()))

for _, row in df.iterrows():
    x0, y0 = pos[row.Drug_A]
    x1, y1 = pos[row.Drug_B]
    nm = abs(row.ZIP) / (vmax_e + 1e-8)
    ax.plot([x0, x1], [y0, y1],
            color=RED if row.ZIP >= 0 else BLUE,
            lw=0.5 + nm * 5.5, alpha=0.3 + nm * 0.6, zorder=1)
    if abs(row.ZIP) > 3.5:
        ax.text((x0+x1)/2, (y0+y1)/2, f'{row.ZIP:.1f}',
                fontsize=9, ha='center', va='center',
                color=DARK, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.1', fc='white',
                          alpha=0.75, ec='none'))

for i, (d, (x, y)) in enumerate(pos.items()):
    ax.add_patch(plt.Circle((x, y), 0.13, color=NC[i],
                             zorder=3, ec='white', lw=2.5))
    ax.text(x*1.35, y*1.35, d, ha='center', va='center',
            fontsize=13, fontweight='bold', color=DARK, zorder=4)

ax.set_xlim(-1.7, 1.7)
ax.set_ylim(-1.7, 1.7)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title(
    'Phytochemical Pairwise Interaction Network (ZIP scores)\n'
    'Edge: Red = Synergistic | Blue = Antagonistic | Width ∝ |ZIP|',
    fontsize=15, fontweight='bold', pad=16
)
from matplotlib.lines import Line2D
ax.legend(
    handles=[Line2D([0],[0], color=RED,  lw=4, label='Synergistic (ZIP ≥ 0)'),
             Line2D([0],[0], color=BLUE, lw=4, label='Antagonistic (ZIP < 0)')],
    loc='lower center', fontsize=10, ncol=2,
    bbox_to_anchor=(0.5, -0.04), framealpha=0.9
)
plt.tight_layout()
for ext in ['png', 'pdf']:
    fig.savefig(f"{OUT}/fig3_network.{ext}",
                dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("  Fig 3 done.")

# ── Fig 4: Four-model comparison, top 8 pairs by ZIP ─────────────────────────
print("Generating Figure 4 — model comparison...")
dt  = df.sort_values('ZIP', ascending=False).head(8).reset_index(drop=True)
fig, ax = plt.subplots(figsize=(15, 8))
x   = np.arange(len(dt))
w   = 0.21
mc  = {'HSA': '#C0392B', 'Bliss': '#8E44AD',
       'Loewe': '#2980B9', 'ZIP': '#27AE60'}
for (m, c), off in zip(mc.items(), [-1.5, -0.5, 0.5, 1.5]):
    vv   = dt[m].values
    bars = ax.bar(x + off*w, vv, w, color=c, alpha=0.88,
                  label=m, ec='white', lw=0.5)
    for bar, v in zip(bars, vv):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.1,
                f'{v:.1f}', ha='center', va='bottom',
                fontsize=10, color=DARK)
ax.axhline(0, color='black', lw=1, zorder=5)
ax.axhline(5, color=GREY, lw=1, ls='--', alpha=0.6)
ax.set_xticks(x)
ax.set_xticklabels(dt.Pair, rotation=35, ha='right', fontsize=12)
ax.set_ylabel('Synergy Score', fontsize=15)
ax.set_title(
    'Synergy Model Comparison — Top 8 Pairs by ZIP Score\n'
    '(HSA | Bliss | Loewe | ZIP  with ChEMBL IC50 values)',
    fontsize=14, fontweight='bold'
)
ax.legend(fontsize=11, loc='upper right', framealpha=0.9)
ax.yaxis.grid(True, color='#EEE', lw=0.8)
ax.set_axisbelow(True)
plt.tight_layout()
for ext in ['png', 'pdf']:
    fig.savefig(f"{OUT}/fig4_model_comparison.{ext}",
                dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("  Fig 4 done.")

# ── Fig 5: Tanimoto similarity + NN training loss ────────────────────────────
print("Generating Figure 5 — similarity + training curve...")
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor('white')

sns.heatmap(
    pd.DataFrame(SIM, index=DRUGS, columns=DRUGS),
    ax=ax1, cmap='YlOrRd', vmin=0, vmax=1,
    annot=True, fmt='.2f', annot_kws={'size': 12},
    linewidths=0.5, linecolor='white', square=True,
    cbar_kws={'label': 'Tanimoto Similarity', 'shrink': 0.85}
)
ax1.set_title(
    'Molecular Fingerprint Similarity\n(CACTVS 881-bit, Tanimoto coefficient)',
    fontsize=15, fontweight='bold'
)
ax1.tick_params(axis='x', rotation=45, labelsize=12)
ax1.tick_params(axis='y', rotation=0,  labelsize=12)

ax2.plot(range(1, len(losses)+1), losses, color=DARK, lw=2)
ax2.fill_between(range(1, len(losses)+1), losses, alpha=0.12, color=DARK)
ax2.set_xlabel('Training Epoch', fontsize=15)
ax2.set_ylabel('MSE Loss (normalised ZIP)', fontsize=15)
ax2.set_title(
    'DeepSynergy NN Training Curve\n'
    '(3-layer feedforward | Input: concatenated fingerprint pairs)',
    fontsize=15, fontweight='bold'
)
ax2.annotate(
    f'Final loss: {losses[-1]:.4f}',
    xy=(len(losses), losses[-1]),
    xytext=(len(losses)*0.55, losses[0]*0.55),
    arrowprops=dict(arrowstyle='->', color=DARK),
    fontsize=12, color=DARK
)
ax2.yaxis.grid(True, color='#EEE', lw=0.7)
ax2.set_axisbelow(True)
plt.tight_layout()
for ext in ['png', 'pdf']:
    fig.savefig(f"{OUT}/fig5_similarity_training.{ext}",
                dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("  Fig 5 done.")

# ── Fig 6: NN predicted vs computed ZIP ──────────────────────────────────────
print("Generating Figure 6 — NN validation scatter...")
fig, ax = plt.subplots(figsize=(10, 9))
ax.scatter(df.ZIP, df.NN_Pred_ZIP,
           s=120, c=DARK, alpha=0.8, ec='white', lw=1.2, zorder=3)
for _, row in df.iterrows():
    ax.annotate(
        row.Pair.replace('–', '\n'),
        (row.ZIP, row.NN_Pred_ZIP),
        textcoords='offset points', xytext=(6, 2),
        fontsize=9, color=GREY
    )
mn = min(df.ZIP.min(), df.NN_Pred_ZIP.min()) - 1
mx = max(df.ZIP.max(), df.NN_Pred_ZIP.max()) + 1
ax.plot([mn, mx], [mn, mx], '--', color=RED, lw=1.5, label='Ideal (y = x)')
ax.set_xlim(mn, mx)
ax.set_ylim(mn, mx)
ax.text(0.06, 0.92, f'Pearson r = {r_pearson:.3f}',
        transform=ax.transAxes, fontsize=14, color=DARK, fontweight='bold',
        bbox=dict(boxstyle='round', fc='#F8F9FA', ec='#BDC3C7', alpha=0.9))
ax.set_xlabel('Computed ZIP Score (Hill model, ChEMBL IC50)', fontsize=15)
ax.set_ylabel('DeepSynergy NN Predicted ZIP', fontsize=15)
ax.set_title(
    'DeepSynergy Neural Network Internal Validation\n'
    'Predicted vs Computed ZIP Synergy Scores',
    fontsize=15, fontweight='bold'
)
ax.legend(fontsize=11)
ax.grid(True, color='#EEE', lw=0.7)
ax.set_axisbelow(True)
plt.tight_layout()
for ext in ['png', 'pdf']:
    fig.savefig(f"{OUT}/fig6_nn_validation.{ext}",
                dpi=300, bbox_inches='tight', facecolor='white')
plt.close()
print("  Fig 6 done.")

# ── Summary ───────────────────────────────────────────────────────────────────
top3 = df.sort_values('ZIP', ascending=False).head(3)
bot3 = df.sort_values('ZIP').head(3)

print("\n" + "="*65)
print("TOP 3 SYNERGISTIC PAIRS (ZIP score)")
print("="*65)
for _, r in top3.iterrows():
    print(f"  {r.Pair:<40} ZIP={r.ZIP:.3f}  HSA={r.HSA:.3f}")

print("\nTOP 3 ANTAGONISTIC PAIRS (ZIP score)")
print("="*65)
for _, r in bot3.iterrows():
    print(f"  {r.Pair:<40} ZIP={r.ZIP:.3f}  HSA={r.HSA:.3f}")

print("\n⚠ TARAXASTEROL REMINDER:")
print("  IC50 = 32,000 nM is a literature estimate with no ChEMBL record.")
print("  Manually cite a published source before submission.")
print("="*65)

IC50 VALUES IN USE (ChEMBL-sourced)
Drug             IC50 (nM)  IC50 (µM)  ChEMBL ID       PMID
---------------------------------------------------------------------------
Mahanine           7,000.0      7.000  CHEMBL590522    23829449
Taraxasterol      32,000.0     32.000  CHEMBL1796000   MANUAL REQUIRED
Tricin             8,000.0      8.000  CHEMBL454320    27955927
Tamarixetin       14,850.0     14.850  CHEMBL226034    25139569
Annomontine       19,120.0     19.120  CHEMBL501413    36931118
Protopine         34,000.0     34.000  CHEMBL486179    20594848
Atractylon        25,100.0     25.100  CHEMBL486189    9544564

Fetching molecular fingerprints from PubChem...
  Mahanine       : 185 bits set
  Taraxasterol   : 68 bits set
  Tricin         : 141 bits set
  Tamarixetin    : 144 bits set
  Annomontine    : 164 bits set
  Protopine      : 161 bits set
  Atractylon     : 120 bits set

Computing pairwise synergy scores (ChEMBL IC50 values)...
  Mahanine–Taraxasterol: ZIP=2.97  HSA=3.01